# Building a Simple LLM with PyTorch: Creating an Inference Pipeline

In this notebook, we'll explore how to use our trained language model for text generation. We'll cover:

1. Loading a trained model
2. Setting up the inference pipeline
3. Different text generation strategies
4. Practical examples and use cases
5. Performance optimization

Let's dive in!

## 1. Setup and Model Loading

First, let's import our dependencies and load our trained model:

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import torch
from src.model import create_model
from src.data import DataModule
from src.inference_pipeline import create_inference_pipeline

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load trained model
save_dir = Path("saved_model")
checkpoint = torch.load(save_dir / 'model.pt', map_location=device)

# Create model with saved config
model = create_model(
    vocab_size=checkpoint['config']['vocab_size'],
    hidden_size=checkpoint['config']['hidden_size'],
    num_layers=checkpoint['config']['num_hidden_layers'],
    num_heads=checkpoint['config']['num_attention_heads']
)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

# Initialize data module to get tokenizer
data_module = DataModule()
data_module.prepare_data()

## 2. Setting Up the Inference Pipeline

Now let's create our inference pipeline and explore its capabilities:

In [ ]:
# Create inference pipeline
pipeline = create_inference_pipeline(
    model=model,
    tokenizer=data_module.tokenizer,
    device=device
)

# Test basic generation
prompt = "The future of artificial intelligence will"

generated_text = pipeline.generate(
    prompt=prompt,
    max_length=50,
    temperature=1.0,
    do_sample=True
)[0]

print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")

## 3. Exploring Different Generation Strategies

Let's compare different text generation strategies:

In [ ]:
def generate_with_strategy(prompt, **kwargs):
    """Generate text with specific strategy and print results."""
    generated = pipeline.generate(prompt=prompt, max_length=50, **kwargs)[0]
    print(f"Strategy: {kwargs}")
    print(f"Generated: {generated}\n")

prompt = "The impact of climate change on global"

print(f"Prompt: {prompt}\n")

# 1. Greedy decoding
generate_with_strategy(prompt, do_sample=False)

# 2. Temperature sampling
generate_with_strategy(prompt, do_sample=True, temperature=0.7)

# 3. Top-K sampling
generate_with_strategy(prompt, do_sample=True, temperature=1.0, top_k=50)

# 4. Nucleus (top-p) sampling
generate_with_strategy(prompt, do_sample=True, temperature=1.0, top_p=0.9)

# 5. Combined top-k and top-p
generate_with_strategy(prompt, do_sample=True, temperature=0.7, top_k=50, top_p=0.9)

## 4. Practical Use Cases

Let's explore some practical applications of our model:

In [ ]:
def generate_examples(task_type: str, prompts: list, **kwargs):
    """Generate examples for a specific task type."""
    print(f"\n{task_type} Examples:")
    print("-" * 50)
    
    for prompt in prompts:
        generated = pipeline.generate(prompt=prompt, **kwargs)[0]
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 50)

# 1. Article Continuation
article_prompts = [
    "Recent advances in quantum computing have",
    "The role of renewable energy in modern"
]

generate_examples(
    "Article Continuation",
    article_prompts,
    max_length=100,
    temperature=0.7,
    top_k=50,
    top_p=0.9
)

# 2. Question Answering
qa_prompts = [
    "Q: What is the theory of relativity? A:",
    "Q: How does photosynthesis work? A:"
]

generate_examples(
    "Question Answering",
    qa_prompts,
    max_length=150,
    temperature=0.6,
    top_p=0.9
)

# 3. Creative Writing
creative_prompts = [
    "Once upon a time in a digital world",
    "The robot's first day of consciousness began with"
]

generate_examples(
    "Creative Writing",
    creative_prompts,
    max_length=200,
    temperature=0.9,
    top_k=100
)

## 5. Performance Optimization

Let's explore some techniques to optimize inference performance:

In [ ]:
import time

def measure_generation_time(prompt, **kwargs):
    """Measure time taken for text generation."""
    start_time = time.time()
    generated = pipeline.generate(prompt=prompt, **kwargs)[0]
    end_time = time.time()
    return end_time - start_time, len(generated.split())

prompt = "The science of machine learning is"

print("Performance Benchmarks:")
print("-" * 50)

# 1. Basic generation
time_taken, word_count = measure_generation_time(
    prompt,
    max_length=100,
    temperature=0.7
)
print(f"Basic Generation: {time_taken:.2f}s for {word_count} words")

# 2. Batch generation
time_taken, word_count = measure_generation_time(
    prompt,
    max_length=100,
    temperature=0.7,
    num_return_sequences=4,
    batch_size=4
)
print(f"Batch Generation: {time_taken:.2f}s for {word_count} words × 4 sequences")

# 3. Optimized settings
time_taken, word_count = measure_generation_time(
    prompt,
    max_length=100,
    temperature=0.7,
    top_k=50,
    top_p=0.9
)
print(f"Optimized Settings: {time_taken:.2f}s for {word_count} words")

## 6. Best Practices and Tips

Here are some key takeaways for using the inference pipeline effectively:

1. **Temperature Control**:
   - Lower temperature (0.3-0.7) for focused, coherent text
   - Higher temperature (0.7-1.0) for more creative, diverse outputs

2. **Sampling Strategies**:
   - Use top-k (k=50) for a good balance of quality and diversity
   - Combine with top-p (p=0.9) for even better results
   - Use greedy decoding (do_sample=False) for deterministic outputs

3. **Performance Tips**:
   - Use batch generation for multiple sequences
   - Keep max_length reasonable (50-200 tokens)
   - Use GPU when available for faster inference

4. **Prompt Engineering**:
   - Be specific and clear in prompts
   - Include context or examples if needed
   - Use consistent formatting for similar tasks

## Next Steps

We've successfully:
1. Set up an inference pipeline
2. Explored different generation strategies
3. Implemented practical use cases
4. Optimized performance

In the next notebook, we'll look at fine-tuning our model for specific tasks and domains.